# Run the complete election pipeline
Evaluate eight candidates across five outer elections, select by equal-weight mean accuracy, then retune, refit and evaluate 2024. Configure PostgreSQL and the tracking server first using docs/postgresql.md and docs/mlflow.md. The pipeline automatically reads .env.election. Source .mlflow/config.env before starting Jupyter for tracking settings. Rerun the setup cell below after updating the database code, including in an existing kernel.

Prepared CSVs, predictor guide, final predictions and confusion matrices remain in notebooks/outputs. Compact experiment histories are retained in MLflow.


In [ ]:
import os
import importlib

# Use the local tracking server unless a server is already configured.
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

from election import paths
from election.sql import apply_sql_queries

# Refresh database setup when this cell is rerun in an existing kernel.
importlib.reload(apply_sql_queries)
from election.pipeline.run_pipeline import run_pipeline

project_root = paths.project_root()

output_directory = project_root / "notebooks" / "outputs"
print(f"Saving pipeline outputs to: {output_directory}")


In [ ]:
# This runs preparation, all historical evaluations, comparison, and final 2024 evaluation.
# To reuse data, pass prepared_data=output_directory / "latest_prepared.json".
# To reuse recorded historical evaluations as well, add reuse_evaluations=True.
selection = run_pipeline(output_dir=output_directory)
import pandas as pd
from IPython.display import display
comparison = selection["comparison"]
final_evaluation = selection["final_evaluation"]
display(pd.DataFrame(comparison["ranked_results"]))
print("Selected model:", comparison["winner_model_id"])
print("Historical source runs:", comparison["source_run_ids"])
print("Final MLflow run:", final_evaluation["run_id"])
print("Prepared data:", selection["prepared_data"])


## Final 2024 results
The pipeline has already evaluated and logged the final model. These cells display returned results and the saved confusion matrix.


In [ ]:
display(pd.Series(final_evaluation["outer_results"][2024], name="2024"))
from IPython.display import Image, display
display(Image(filename=str(output_directory / "test_confusion_matrix.png")))
